<a href="https://colab.research.google.com/github/hidrofluorik/Dataset-Google-Play-Store-Apps/blob/main/Tugas_Kelompok_K_Means_PlayStore2_Anggota1_Revisi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyek Akhir Pembelajaran Mesin
## Segmentasi Aplikasi Google Play Store Menggunakan K-Means
### Anggota 1 — Varian Dasar

Notebook ini menjalankan varian dasar K-Means menggunakan fitur **Rating, Reviews, dan Installs tanpa transformasi logaritma (`log1p`)**. Sumber dataset dan prosedur pembersihan disamakan dengan notebook anggota 2 agar hasil eksperimen kelompok menggunakan data yang konsisten.

**Catatan eksperimen anggota 1:** fitur tidak ditransformasi dengan `log1p`, tetapi tetap distandardisasi menggunakan `StandardScaler` sebelum K-Means.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 1. Memuat Dataset

Gunakan dataset asli `googleplaystore.csv` yang sama dengan anggota 2 dan anggota 3. Dataset asli memiliki 13 kolom. Pastikan file CSV tersebut sudah diunggah ke sesi Google Colab.

In [ ]:
# Memuat dataset asli yang sama dengan anggota 2 dan anggota 3.
df = pd.read_csv('/content/googleplaystore.csv')
df.columns = df.columns.str.strip()

print("Dataset berhasil dimuat.")
print("Ukuran dataset:", df.shape)
print("Daftar kolom:", df.columns.tolist())
display(df.head())

## 2. Pemeriksaan Awal Dataset (EDA)

Pemeriksaan awal dilakukan sebelum pembersihan data untuk mengetahui jumlah nilai kosong, jumlah baris duplikat, tipe data, dan struktur dataset.

In [ ]:
print("=== Informasi Dataset ===")
df.info()

print("\n=== Jumlah Nilai Kosong per Kolom ===")
print(df.isnull().sum())
print("\nTotal nilai kosong:", df.isnull().sum().sum())

print("\nJumlah baris duplikat identik:", df.duplicated().sum())

## 3. Pembersihan Dataset (Data Cleaning)

Tahapan pembersihan dibuat sama dengan notebook anggota 2 agar seluruh eksperimen menggunakan dasar data yang konsisten:

1. Menghapus baris duplikat identik.
2. Mengubah `Rating` menjadi numerik dan menghapus baris dengan `Rating` kosong.
3. Menghapus nilai `Rating` yang tidak valid, yaitu di luar rentang 1–5.
4. Menghapus baris yang memiliki nilai kosong pada `Type`, `Content Rating`, `Current Ver`, dan `Android Ver`.

Nilai `Rating` kosong **tidak diisi dengan median**. Perlakuan ini mengikuti notebook anggota 2.

In [ ]:
# Salinan dataset untuk proses cleaning.
df_clean = df.copy()

# Pastikan Rating bertipe numerik.
df_clean['Rating'] = pd.to_numeric(
    df_clean['Rating'], errors='coerce'
)

# 1. Menghapus baris duplikat identik.
df_clean = df_clean.drop_duplicates().copy()

# 2. Menghapus baris dengan Rating kosong.
df_clean = df_clean.dropna(subset=['Rating']).copy()

# 3. Menghapus Rating yang tidak valid (di luar rentang 1 sampai 5).
df_clean = df_clean[
    df_clean['Rating'].between(1, 5)
].copy()

# 4. Menghapus nilai kosong pada kolom lain, sama seperti anggota 2.
columns_to_check = [
    col for col in ['Type', 'Content Rating', 'Current Ver', 'Android Ver']
    if col in df_clean.columns
]
if columns_to_check:
    df_clean = df_clean.dropna(subset=columns_to_check).copy()

print("=== Hasil Pembersihan ===")
print("Ukuran dataset setelah cleaning:", df_clean.shape)
print("Jumlah nilai kosong seluruh kolom:", df_clean.isnull().sum().sum())
print("Jumlah baris duplikat tersisa:", df_clean.duplicated().sum())
print("Kolom yang diperiksa untuk nilai kosong:", columns_to_check)

# Pemeriksaan konsistensi dengan hasil cleaning anggota 2.
print("\nJumlah data yang diharapkan dari prosedur cleaning bersama: 8.886 baris")
print("Jumlah data aktual:", len(df_clean))

## 4. Persiapan Fitur untuk K-Means

Sesuai topik penelitian dan pembagian tugas, anggota 1 menggunakan tiga fitur:

- `Rating`: nilai penilaian aplikasi pada rentang 1–5.
- `Reviews`: jumlah ulasan pengguna.
- `Installs`: jumlah pemasangan aplikasi.

Kolom `Installs` dibersihkan dari tanda koma dan tanda `+`, lalu dikonversi menjadi numerik. Pada varian dasar ini, **Reviews dan Installs tidak diberi transformasi `log1p`**.

In [ ]:
# Mengambil tiga fitur yang digunakan pada varian dasar.
fitur = ['Rating', 'Reviews', 'Installs']
df_kmeans = df_clean[fitur].copy()

# Mengubah fitur menjadi numerik.
for kolom in ['Rating', 'Reviews']:
    df_kmeans[kolom] = pd.to_numeric(
        df_kmeans[kolom], errors='coerce'
    )

# Membersihkan format Installs (contoh: '10,000+') sebelum konversi.
df_kmeans['Installs'] = (
    df_kmeans['Installs'].astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)
df_kmeans['Installs'] = pd.to_numeric(
    df_kmeans['Installs'], errors='coerce'
)

# Menghapus baris yang tidak valid pada fitur yang digunakan.
df_kmeans = df_kmeans.dropna(subset=fitur).copy()
df_kmeans = df_kmeans[
    df_kmeans['Rating'].between(1, 5)
    & (df_kmeans['Reviews'] >= 0)
    & (df_kmeans['Installs'] >= 0)
].reset_index(drop=True)

print("Jumlah data untuk K-Means:", len(df_kmeans))
print("Jumlah fitur:", len(fitur))
print("Nilai kosong per fitur:")
print(df_kmeans.isnull().sum())
print("\nStatistik deskriptif:")
display(df_kmeans.describe())

## 5. Eksplorasi Data Fitur (EDA)

Distribusi ketiga fitur diperiksa untuk memahami karakteristik data sebelum pemodelan. Reviews dan Installs dapat memiliki distribusi yang menceng ke kanan (*right-skewed*) dan rentang nilai yang lebar.

Pada **varian dasar anggota 1**, distribusi tersebut tetap digunakan dalam bentuk aslinya. Tidak ada transformasi logaritma pada fitur model. Standardisasi dilakukan pada tahap berikutnya agar skala fitur sebanding.

In [ ]:
# Visualisasi distribusi fitur sebelum pemodelan.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df_kmeans['Rating'], bins=20, edgecolor='black', alpha=0.75)
axes[0].set_title('Distribusi Rating')

axes[1].hist(df_kmeans['Reviews'], bins=30, edgecolor='black', alpha=0.75)
axes[1].set_title('Distribusi Reviews (tanpa log1p)')

axes[2].hist(df_kmeans['Installs'], bins=30, edgecolor='black', alpha=0.75)
axes[2].set_title('Distribusi Installs (tanpa log1p)')

plt.tight_layout()
plt.show()

print("Skewness setiap fitur:")
print(df_kmeans[fitur].skew())

In [ ]:
# Matriks korelasi untuk tiga fitur yang digunakan.
corr_matrix = df_kmeans[fitur].corr()
print("Korelasi antarfitur:")
display(corr_matrix)

plt.figure(figsize=(6, 4))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='Blues',
    fmt='.3f',
    linewidths=0.5
)
plt.title('Matriks Korelasi: Rating, Reviews, dan Installs')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot hubungan antarfitur dalam skala asli.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(
    df_kmeans['Reviews'], df_kmeans['Installs'],
    alpha=0.35, s=14
)
axes[0].set_title('Reviews vs Installs (Skala Asli)')
axes[0].set_xlabel('Reviews')
axes[0].set_ylabel('Installs')

axes[1].scatter(
    df_kmeans['Rating'], df_kmeans['Installs'],
    alpha=0.35, s=14
)
axes[1].set_title('Rating vs Installs (Skala Asli)')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Installs')

plt.tight_layout()
plt.show()

## 6. Standardisasi Fitur

K-Means menggunakan jarak antar-data. Karena Rating, Reviews, dan Installs memiliki rentang nilai yang berbeda, ketiga fitur distandardisasi menggunakan `StandardScaler`.

**Penting:** Standardisasi bukan transformasi logaritma. Nilai fitur tetap berasal dari data asli (tanpa `log1p`), hanya skalanya yang disetarakan.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_kmeans[fitur])

print("Ukuran data setelah standardisasi:", X_scaled.shape)
print("Rata-rata setiap fitur:", X_scaled.mean(axis=0).round(3))
print("Standar deviasi setiap fitur:", X_scaled.std(axis=0).round(3))

## 7. Menentukan Jumlah Cluster dengan Elbow Method

Elbow Method dijalankan untuk jumlah cluster `K` dari 2 sampai 10. Pada setiap nilai K, nilai inertia dicatat. Kandidat jumlah cluster dipilih dengan mengamati titik ketika penurunan inertia mulai melandai.

In [ ]:
k_values = range(2, 11)
inertia_values = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)

elbow_results = pd.DataFrame({
    'Jumlah Cluster (K)': list(k_values),
    'Inertia': inertia_values
})

display(elbow_results)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertia_values, marker='o')
plt.title('Elbow Method — Varian Dasar (Tanpa log1p)')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Inertia')
plt.xticks(list(k_values))
plt.grid(True)
plt.tight_layout()
plt.show()

### Interpretasi Elbow Method

Amati grafik dan tentukan titik siku (*elbow*), yaitu saat penurunan inertia yang awalnya tajam mulai melandai. Catat kandidat K berdasarkan grafik aktual setelah notebook dijalankan. Jangan menetapkan titik siku tanpa melihat hasil grafik.

## 8. Evaluasi Silhouette Score

Silhouette Score dihitung untuk K=2 sampai K=10 sebagai evaluasi tambahan terhadap kualitas pemisahan cluster. Perhitungan menggunakan data yang sudah distandardisasi, tetapi **belum ditransformasi dengan `log1p`**.

In [ ]:
hasil_silhouette = []

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)

    hasil_silhouette.append({
        'K': k,
        'Silhouette Score': score
    })

df_silhouette = pd.DataFrame(hasil_silhouette)
display(df_silhouette)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    df_silhouette['K'],
    df_silhouette['Silhouette Score'],
    marker='o'
)
plt.title('Silhouette Score — Varian Dasar (Tanpa log1p)')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Silhouette Score')
plt.xticks(range(2, 11))
plt.grid(True)
plt.tight_layout()
plt.show()

baris_terbaik = df_silhouette.loc[
    df_silhouette['Silhouette Score'].idxmax()
]
print(
    f"Silhouette Score tertinggi diperoleh pada K={int(baris_terbaik['K'])} "
    f"dengan skor {baris_terbaik['Silhouette Score']:.4f}."
)

### Interpretasi Silhouette Score dan Pemilihan K

Gunakan tabel dan grafik Silhouette Score bersama grafik Elbow Method untuk menentukan jumlah cluster yang layak. Silhouette Score yang lebih tinggi menunjukkan pemisahan cluster yang lebih baik menurut metrik tersebut. Keputusan akhir tetap perlu mempertimbangkan bentuk grafik Elbow dan kemudahan interpretasi cluster.

## 9. Clustering Akhir untuk Perbandingan Kelompok

Untuk menjaga perbandingan dengan eksperimen anggota 2 dan anggota 3, notebook ini juga menjalankan model dasar pada `K=3` (jumlah cluster yang digunakan pada perbandingan kelompok). Bagian ini tidak mengubah eksperimen Elbow dan Silhouette K=2–10 di atas; K=3 digunakan sebagai konfigurasi bersama untuk melihat profil cluster.

In [ ]:
K_FINAL = 3

model_dasar = KMeans(
    n_clusters=K_FINAL,
    random_state=42,
    n_init=10
)
label_cluster = model_dasar.fit_predict(X_scaled)

# Simpan label cluster bersama nilai fitur asli.
df_hasil = df_kmeans.copy()
df_hasil['Cluster'] = label_cluster

score_k3 = silhouette_score(X_scaled, label_cluster)
print(f"Silhouette Score model dasar pada K={K_FINAL}: {score_k3:.4f}")
print("Jumlah anggota setiap cluster:")
print(df_hasil['Cluster'].value_counts().sort_index())

In [ ]:
# Profil setiap cluster menggunakan nilai fitur asli (tanpa log1p).
profil_cluster = (
    df_hasil.groupby('Cluster')[fitur]
    .agg(['mean', 'median', 'count'])
)

display(profil_cluster)

In [ ]:
# Visualisasi cluster pada dua fitur utama.
plt.figure(figsize=(8, 5))
plt.scatter(
    df_hasil['Reviews'],
    df_hasil['Installs'],
    c=df_hasil['Cluster'],
    alpha=0.45,
    s=14
)
plt.title('Hasil K-Means Varian Dasar: Reviews vs Installs (K=3)')
plt.xlabel('Reviews')
plt.ylabel('Installs')
plt.tight_layout()
plt.show()

## 10. Kesimpulan Sementara Anggota 1

Tuliskan kesimpulan setelah seluruh sel dijalankan. Kesimpulan minimal mencakup:

1. Jumlah data yang digunakan setelah cleaning dan preprocessing.
2. Kandidat jumlah cluster berdasarkan Elbow Method.
3. Nilai Silhouette Score tertinggi beserta nilai K-nya.
4. Nilai Silhouette Score model dasar pada K=3 untuk perbandingan kelompok.
5. Karakteristik tiap cluster berdasarkan rata-rata/median Rating, Reviews, dan Installs.

**Pengingat:** Anggota 1 menggunakan Rating, Reviews, dan Installs tanpa transformasi `log1p`. Jangan menyalin kesimpulan anggota 2 karena perlakuan datanya berbeda.